<a href="https://colab.research.google.com/github/pramodjella/QuantumML-IIT-Delhi-/blob/main/ML_Vs_QML_pennylane_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit qiskit-algorithms qiskit-machine-learning qiskit-aer-gpu datasets seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.1 

In [ ]:
"""
Quantum vs Classical Sentiment Analysis: GPU Accelerated (Fixed & Optimized)
------------------------------------------------------
Features:
1. Fixes 'unknown instruction: ZZFeatureMap' by decomposing circuits.
2. Uses T4 GPU via qiskit-aer SamplerV2.
3. Complete Train/Val/Test pipeline with visualizations.
"""

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data & ML Imports
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Qiskit Imports
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.algorithms import VQC, QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# GPU Simulator Import (V2)
try:
    from qiskit_aer import AerSimulator
    from qiskit_aer.primitives import SamplerV2 as AerSampler
    HAS_GPU_LIB = True
except ImportError:
    from qiskit.primitives import StatevectorSampler
    HAS_GPU_LIB = False
    print("WARNING: qiskit-aer-gpu not found. Running on CPU (Slow).")

# Configuration
algorithm_globals.random_seed = 42
np.random.seed(42)
sns.set_theme(style="whitegrid")

class GPUQuantumComparison:
    def __init__(self, n_features=4):
        self.n_features = n_features
        # Data sizes to experiment with
        self.data_sizes = {
            'Small': 400,
            'Medium': 800,
            'Large': 1600
        }
        self.results_df = pd.DataFrame()
        self.sampler = self._setup_sampler()

    def _setup_sampler(self):
        """Configures the Quantum Sampler (GPU V2 if available)"""
        if HAS_GPU_LIB:
            try:
                # Check availability
                sim = AerSimulator()
                if 'GPU' in sim.available_devices():
                    print(f"🚀 GPU ACCELERATION ENABLED: {sim.available_devices()}")

                    # Initialize V2 Sampler
                    sampler = AerSampler()

                    # Configure GPU options for V2
                    # Note: V2 uses 'default_shots' but for statevector we often want precision
                    sampler.options.default_shots = None  # Use exact probabilities if possible

                    # Inject backend options directly (Aer specific)
                    # This tells Aer to use the GPU backend
                    sampler.options.backend_options = {
                        "method": "statevector",
                        "device": "GPU"
                    }
                    return sampler
                else:
                    print("⚠️  GPU not detected. Using Aer CPU V2 Sampler.")
                    return AerSampler()
            except Exception as e:
                print(f"⚠️ Error initializing GPU sampler: {e}. Falling back to CPU.")
                from qiskit.primitives import StatevectorSampler
                return StatevectorSampler(seed=42)
        else:
            print("⚠️ Standard CPU Fallback.")
            from qiskit.primitives import StatevectorSampler
            return StatevectorSampler(seed=42)

    def load_data(self, n_samples):
        print(f"\n... Processing Data Batch (N={n_samples}) ...")
        dataset = load_dataset('glue', 'sst2')

        # 1. Prepare Training Pool
        raw_train_texts = dataset['train']['sentence'][:n_samples]
        raw_train_labels = dataset['train']['label'][:n_samples]

        # 2. Split Training Pool -> Train (80%) + Validation (20%)
        X_train_txt, X_val_txt, y_train, y_val = train_test_split(
            raw_train_texts, raw_train_labels, test_size=0.2, random_state=42, stratify=raw_train_labels
        )

        # 3. Prepare Test Set
        X_test_txt = dataset['validation']['sentence'][:200]
        y_test = dataset['validation']['label'][:200]

        # 4. Vectorize (TF-IDF)
        vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
        X_train_vec = vectorizer.fit_transform(X_train_txt).toarray()
        X_val_vec = vectorizer.transform(X_val_txt).toarray()
        X_test_vec = vectorizer.transform(X_test_txt).toarray()

        # 5. PCA Reduction
        pca = PCA(n_components=self.n_features)
        X_train_pca = pca.fit_transform(X_train_vec)
        X_val_pca = pca.transform(X_val_vec)
        X_test_pca = pca.transform(X_test_vec)

        # 6. Normalize to [0, 1]
        def normalize(data, ref):
            return (data - ref.min()) / (ref.max() - ref.min() + 1e-8)

        X_train_q = normalize(X_train_pca, X_train_pca)
        X_val_q = normalize(X_val_pca, X_train_pca)
        X_test_q = normalize(X_test_pca, X_train_pca)

        # 7. Convert to NumPy Arrays
        y_train = np.array(y_train)
        y_val = np.array(y_val)
        y_test = np.array(y_test)

        return (X_train_pca, y_train, X_val_pca, y_val, X_test_pca, y_test), \
               (X_train_q, y_train, X_val_q, y_val, X_test_q, y_test)

    def train_evaluate(self, model, name, size_name, data_pack, is_vqc=False):
        X_train, y_train, X_val, y_val, X_test, y_test = data_pack

        print(f"  Training {name:<15} ...", end=" ", flush=True)
        start_time = time.time()

        if is_vqc:
            # One-Hot Encoding for VQC
            encoder = OneHotEncoder(sparse_output=False)
            y_train_fit = encoder.fit_transform(y_train.reshape(-1, 1))
            model.fit(X_train, y_train_fit)
        else:
            model.fit(X_train, y_train)

        duration = time.time() - start_time
        print(f"Done ({duration:.2f}s)")

        # Predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)

        # Decode One-Hot if needed
        if is_vqc and len(y_test_pred.shape) > 1:
            y_train_pred = np.argmax(y_train_pred, axis=1)
            y_val_pred = np.argmax(y_val_pred, axis=1)
            y_test_pred = np.argmax(y_test_pred, axis=1)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_test_pred, average='weighted', zero_division=0
        )

        return {
            'Data Size': size_name,
            'Model': name,
            'Train Samples': len(X_train),
            'Train Acc': accuracy_score(y_train, y_train_pred),
            'Val Acc': accuracy_score(y_val, y_val_pred),
            'Test Acc': accuracy_score(y_test, y_test_pred),
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1,
            'Time (s)': duration
        }

    def run(self):
        all_results = []

        for size_name, n_samples in self.data_sizes.items():
            # Load Data
            c_data, q_data = self.load_data(n_samples)

            # --- Classical Models ---
            lr = LogisticRegression(max_iter=1000)
            all_results.append(self.train_evaluate(lr, "Classical LR", size_name, c_data))

            svc = SVC(kernel='rbf')
            all_results.append(self.train_evaluate(svc, "Classical SVC", size_name, c_data))

            # --- Quantum Setup ---
            # FIX: Decompose high-level circuit objects into basic gates for Aer
            feature_map = ZZFeatureMap(feature_dimension=self.n_features, reps=2, entanglement='linear').decompose()
            ansatz = RealAmplitudes(num_qubits=self.n_features, reps=3, entanglement='linear').decompose()

            # --- Quantum VQC (GPU) ---
            optimizer = COBYLA(maxiter=100)
            vqc = VQC(
                sampler=self.sampler,
                feature_map=feature_map,
                ansatz=ansatz,
                optimizer=optimizer
            )
            all_results.append(self.train_evaluate(vqc, "Quantum VQC", size_name, q_data, is_vqc=True))

            # --- Quantum QSVC (GPU) ---
            fidelity = ComputeUncompute(sampler=self.sampler)
            kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
            qsvc = QSVC(quantum_kernel=kernel)
            all_results.append(self.train_evaluate(qsvc, "Quantum QSVC", size_name, q_data, is_vqc=False))

        self.results_df = pd.DataFrame(all_results)
        return self.results_df

    def visualize_results(self):
        df = self.results_df
        if df.empty: return

        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.2)

        # Plot 1: Test Accuracy
        ax1 = fig.add_subplot(gs[0, 0])
        sns.lineplot(data=df, x='Data Size', y='Test Acc', hue='Model',
                     marker='o', markersize=10, linewidth=3, ax=ax1)
        ax1.set_title('Test Accuracy (Generalization)', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3)

        # Plot 2: Training Time
        ax2 = fig.add_subplot(gs[0, 1])
        sns.barplot(data=df, x='Data Size', y='Time (s)', hue='Model', ax=ax2, palette='muted')
        ax2.set_title('Training Time (Log Scale)', fontsize=14, fontweight='bold')
        ax2.set_yscale('log')
        ax2.grid(True, axis='y', alpha=0.3)

        # Plot 3: Metrics Breakdown
        largest_size = list(self.data_sizes.keys())[-1]
        metrics_df = df[df['Data Size'] == largest_size].melt(
            id_vars=['Model'], value_vars=['Precision', 'Recall', 'F1 Score'],
            var_name='Metric', value_name='Score'
        )

        ax3 = fig.add_subplot(gs[1, 0])
        sns.barplot(data=metrics_df, x='Model', y='Score', hue='Metric', ax=ax3, palette='viridis')
        ax3.set_title(f'Detailed Metrics ({largest_size} Dataset)', fontsize=14, fontweight='bold')
        ax3.set_ylim(0, 1.0)
        ax3.legend(loc='lower right')

        # Plot 4: Overfitting Check
        acc_df = df[df['Data Size'] == largest_size].melt(
            id_vars=['Model'], value_vars=['Train Acc', 'Test Acc'],
            var_name='Type', value_name='Accuracy'
        )

        ax4 = fig.add_subplot(gs[1, 1])
        sns.barplot(data=acc_df, x='Model', y='Accuracy', hue='Type', ax=ax4, palette='RdBu')
        ax4.set_title(f'Overfitting Analysis: Train vs Test ({largest_size} Data)', fontsize=14, fontweight='bold')
        ax4.set_ylim(0, 1.1)

        plt.suptitle(f'Quantum vs Classical NLP: GPU-Accelerated Analysis', fontsize=18, fontweight='bold', y=0.98)
        plt.savefig('gpu_quantum_results.png')
        print("\n📊 Visualization saved to 'gpu_quantum_results.png'")
        plt.show()

if __name__ == "__main__":
    print("="*60)
    print("STARTING GPU QUANTUM EXPERIMENT (V2 Primitives + Decomposed Circuits)")
    print("="*60)

    experiment = GPUQuantumComparison(n_features=4)
    results = experiment.run()

    print("\n" + "="*80)
    print("FINAL RESULTS SUMMARY")
    print("="*80)

    display_cols = ['Data Size', 'Model', 'Train Acc', 'Val Acc', 'Test Acc', 'F1 Score', 'Time (s)']
    print(results[display_cols].to_string(index=False, float_format="%.4f"))

    experiment.visualize_results()

STARTING GPU QUANTUM EXPERIMENT (V2 Primitives + Decomposed Circuits)
⚠️ Standard CPU Fallback.

... Processing Data Batch (N=400) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

  Training Classical LR    ... Done (0.01s)
  Training Classical SVC   ... Done (0.02s)


  Training Quantum VQC     ... 